In [2]:
import sqlite3 # Importing sqlite to connect to kaggle files
import pandas as pd # Importing pandas to convert tables as dataframes

In [3]:
con = sqlite3.connect("../data/bdb2027.sqlite") # Used sqlite3 to connect to database
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))  # list tables

                          name
0              combine_results
1           game_tracking_2024
2             combine_tracking
3           game_tracking_2025
4           game_tracking_2023
5                      players
6                        games
7                  player_play
8      player_career_successes
9            game_tracking_all
10                  df_players
11         df_combine_tracking
12                    df_games
13  df_player_career_successes


In [4]:
df_players = pd.read_sql("SELECT * FROM players", con) # Reading the Players database
df_players.head() # Generated the first few rows of the players database

,nfl_id,display_name,draft_year,nfl_position,birth_date,college_name,college_conference,draft_round,draft_pick_within_round,draft_overall_pick
0,55867,Will Anderson,2023,DE,2001-09-02,Alabama,Southeastern Conference,1.0,3.0,3.0
1,55870,Paris Johnson,2023,T,2001-07-31,Ohio State,Big Ten Conference,1.0,6.0,6.0
2,55874,Darnell Wright,2023,T,2001-08-10,Tennessee,Southeastern Conference,1.0,10.0,10.0
3,55875,Peter Skoronski,2023,G,2001-07-31,Northwestern,Big Ten Conference,1.0,11.0,11.0
4,55877,Lukas Van Ness,2023,DE,2001-07-06,Iowa,Big Ten Conference,1.0,13.0,13.0


In [5]:
df_players.dtypes # Checking data types for each column

df_players['birth_date']=pd.to_datetime(df_players['birth_date'], format='%Y-%m-%d') # Converted the birth_date column from a str to a datetime format
float_columns=['draft_round','draft_pick_within_round','draft_overall_pick'] #Declaring float_columns which contains columns in dataframe that are floats

df_players[float_columns]=df_players[float_columns].astype('Int64') # Converted all float_columns to integers as draft picks are integers instead of float


df_players['drafted']=df_players['draft_round'].notna().astype(int) # Created a drafted column that converts all draft_round picks to indicate if a player was drafted (1) or not drafted(0)

df_players['draft_round']=df_players['draft_round'].fillna(8) #There are 7 draft rounds, so all NA values are filled as 8 indicating theey were not drafted
df_players['draft_pick_within_round']=df_players['draft_pick_within_round'].fillna(44) #There were 43 draft picks within a round rounds, so all NA values are filled as 44 indicating they were not drafted
df_players['draft_overall_pick']=df_players['draft_overall_pick'].fillna(300) # For draft_overall_pick, used the number 300 to indicate a player was not drafted

df_players.head() # Showing the first few rows to show the cleaned dataset

# Converted the cleaned dataset to sql_db and added it to bdb2027.sqlite in dataset
df_players.to_sql(name="df_players", con=con, if_exists="replace", index=False)


510

In [6]:
df_combine_tracking = pd.read_sql("SELECT * FROM combine_tracking", con) #Loading the combine_tracking dataset
df_combine_tracking['time']=pd.to_datetime(df_combine_tracking['time'], format='ISO8601') # Converted the time column to datetime using an ISO8601 format from a str

df_combine_tracking[['x', 'y', 's', 'a', 'dis', 'dir']].describe() #For the float columns aka statistics, used the describe function to check for outliers
fast = df_combine_tracking[df_combine_tracking['s'] > 12] #Checking outlier values in the s column (player speed in yards/second) to determine why were there outliers in this column
fast['entity_type'].value_counts() #Checking the values of the entity (player or ball) and found the high 's' column was referring to ball speed which made sense
df_combine_tracking[df_combine_tracking['entity_type'] == 'PLAYER']['s'].max() #Checking the max player speed to determine whether there were any outliers and there were none
df_combine_tracking.duplicated(subset=['event_id', 'entity_type', 'nfl_id', 'time']).sum() # Checking any duplicated values in these columns and there were none

df_combine_tracking.head() # Showing the first few rows of the cleaned dataset

# Converted the cleaned dataset to sql_db and added it to bdb2027.sqlite in dataset
df_combine_tracking.to_sql(name="df_combine_tracking", con=con, if_exists="replace", index=False)


463189

In [7]:
df_games = pd.read_sql("SELECT * FROM games", con) # Reading the games column using read_sql

df_games.duplicated().sum()  #Checking if there are any duplicate values and there were none
df_games.isna().sum() # Checking for any N/A values and there were none
df_games.groupby('season_type')['week'].agg(['min','max']) #For each season type (regular season or playoffs (post)), deteermining no outliers of the number of weeks in NFL Season
df_games[(df_games['season_type'] == 'POST') & (df_games['week'] >= 22)] # Observed why there were additional weeks during the playoff season (last week was SuperBowl)

#Checking the number of home and visitor teams in NFL were the same (Theree were 33 teams, when the NFL only has 32 teams)
df_games['home_team_abbr'].value_counts().tail() 
df_games['visitor_team_abbr'].value_counts().tail()

# We found that there were 2 LA Values, LAR (LA Rams) and LA (Chargers), and replaced LA with LAC to reflect this
df_games['home_team_abbr']=df_games['home_team_abbr'].replace("LA","LAC")
df_games['visitor_team_abbr']=df_games['visitor_team_abbr'].replace("LA","LAC")

#Final Check of the number of home and visitor teams in the NFL matches (32 teams, which matches the numbere of teams in the NFL)
df_games['home_team_abbr'].nunique()
df_games['visitor_team_abbr'].nunique()

# Checking that there were no duplicate games added in the table and there were none
df_games['game_id'].duplicated().sum()
(df_games['home_team_abbr']==df_games['visitor_team_abbr']).sum()

#Printed cleaned df_games dataset
df_games.head()

# Converted the cleaned dataset to sql_db and added it to bdb2027.sqlite in dataset
df_games.to_sql(name='df_games',con=con,if_exists="replace",index=False)


1002

In [8]:
# Read the player_career_successes dataset from SQL DB
df_player_career_successes = pd.read_sql("SELECT * FROM player_career_successes", con)

# Checked the dataframe for any NaN values or duplicate values and found 0 values for both
df_player_career_successes.isna().sum()
df_player_career_successes.duplicated().sum()

# Checked to see if the the number of career_games_started was greater than the number of regular season games the player was on the active roster
# We found 0 values of this
(df_player_career_successes['career_games_started'] > df_player_career_successes['career_games_active']).sum()

#Checked if there were duplicate player ids and there were none or 0
df_player_career_successes['nfl_id'].duplicated().sum()

# Used the describe function to determine any outliers or weird values for each column, determined that all values were normal
df_player_career_successes.describe()

# Converted the cleaned dataset to sql_db and added it to bdb2027.sqlite in dataset
df_player_career_successes.to_sql(name='df_player_career_successes',con=con,if_exists="replace",index=False)


510

In [9]:
# Read the combine_results dataset from SQL DB
df_combine_results = pd.read_sql("SELECT * FROM combine_results", con)

# Checked the dataframe for any NaN values or duplicate values and found 0 duplicated values
df_combine_results.isna().sum() # A lot of NaN values-- analyzing what is wrong
df_combine_results.duplicated().sum() # no duplicated values

df_combine_results['ngs_college_production_score'].isna() #Checking ngs_college_production_score to determine any NA Values
df_combine_results['ngs_final_score'].isna() #Checking ngs_final_score to determine any NA Values

df_combine_results=df_combine_results[df_combine_results['nfl_id'] != 56040] #Got rid of player id: 56040 from data, as he was the only player to have tons of N/A Values

df_combine_results.isna().mean() #Checked the N/A Values, determined to keep in the data as the drills they were asked to do they opted out of according to Kaggle

df_combine_results['combine_position'].value_counts() #Found the counts of combine_position and found nothing significant

df_combine_results.describe() #Used describe function and found no outliers that could impact the data

# Converted the cleaned dataset to csv and added it to processed folder in dataset
df_combine_results.to_sql(name='df_combine_results', con=con, if_exists="replace", index=False)



509

In [10]:
# For game_tracking, there are 3 files corresponding for each year (2023,2024, 2025), rather than
# combine them which would have resulted in excess processing time using pd.concat, (Combined file was ~ 2 GB),
# I decided to creaete a table called game_tracking_all combining all tables using a SQL query and executed them.

query = """
    CREATE TABLE game_tracking_all AS
    SELECT *, 2023 AS season FROM game_tracking_2023
    UNION ALL
    SELECT *, 2024 AS season FROM game_tracking_2024
    UNION ALL
    SELECT *, 2025 AS season FROM game_tracking_2025
"""
con.execute(query)
con.commit()

OperationalError: table game_tracking_all already exists

In [12]:
# For each season, we are tracking how many rows corresponding for each season aka verifying
# accuracy and completneesess for each season, and we confirmed from the dataset on Kaggle that
# all values here match what we found.

pd.read_sql("""
    SELECT season, COUNT(*) AS n_rows
    FROM game_tracking_all
    GROUP BY season
""", con)



,season,n_rows
0,2023,3665443
1,2024,7827944
2,2025,11160903


In [13]:
# Creaeted a new sql query, where we subset certain columns. For each player (nfl_id) and season,
# we track the player ID, how many games playede that were tracked, what their max speed was, average speed was, maximum acceleration,
# total_distance, and distance per gamee travelled.

df_tracking_summary_year = pd.read_sql("""
    SELECT nfl_id,
           season,
           COUNT(DISTINCT game_id)                  AS games_tracked,
           MAX(s)                                   AS max_speed,
           AVG(s)                                   AS avg_speed,
           MAX(a)                                   AS max_accel,
           SUM(dis)                                 AS total_distance,
           SUM(dis) / COUNT(DISTINCT game_id)       AS distance_per_game
    FROM game_tracking_all
    GROUP BY nfl_id, season
""", con)



In [14]:
df_tracking_summary_year.isna().sum() #Checked if there were any NaN values, and confirmed 0
df_tracking_summary_year.duplicated(subset=['nfl_id','season']).sum() #Tracking any duplicate player ids (nfl_id) and found 0
df_tracking_summary_year.describe() # Used a describe function to determine summary statistics and found
# that that the max_speed of 16.88 yd/s and an acceleration of 33.76 yd/sec^2 is impossible for a player

,nfl_id,season,games_tracked,max_speed,avg_speed,max_accel,total_distance,distance_per_game
count,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000
mean,56946.091106,2024.316703,11.341649,8.605531,2.692091,7.227057,6237.469334,491.119105
std,920.409160,0.745251,6.802581,1.476685,0.788885,2.389567,5654.939191,250.562492
min,55867.000000,2023.000000,1.000000,4.710000,1.319528,2.750000,37.360000,37.360000
25%,56027.000000,2024.000000,4.000000,7.510000,1.930697,5.562500,1574.587500,292.876798
50%,57154.500000,2024.000000,12.000000,8.765000,2.730468,7.185000,4041.355000,447.543333
75%,57371.000000,2025.000000,18.000000,9.747500,3.351864,8.360000,10203.782500,666.879912
max,59278.000000,2025.000000,24.000000,16.880000,5.440426,33.759000,28595.170000,1307.901053


In [15]:
df_tracking_summary = pd.read_sql("""
    SELECT nfl_id,
           season,
           COUNT(DISTINCT game_id)              AS games_tracked,
           MAX(s)                               AS max_speed,
           AVG(s)                               AS avg_speed,
           MAX(a)                               AS max_accel,
           SUM(dis)                             AS total_distance,
           SUM(dis) / COUNT(DISTINCT game_id)   AS distance_per_game
    FROM game_tracking_all
    WHERE s <= 12 AND a <= 12   -- remove physically impossible frames (tracking glitches)
    GROUP BY nfl_id, season
""", con) # Created a new dataframe called df_tracking_summary and filtered out thesee outliers
# with the same results and were filtered 



In [16]:
df_tracking_summary_year.describe() #Checked summary stats again and found no weird outliers
df_tracking_summary_year.to_sql(name='df_tracking_summary_year', con=con, if_exists="replace", index=False) # Converted dataframe to SQL db


922

In [17]:
df_player_play= pd.read_sql("SELECT * FROM player_play", con) #Using read_sql to read in player_play file

df_player_play.isna().sum() #Tracking the dataframe for any NaN values and we found a lot

# Found a high proportion of NaN values but decided to keep as some of the columns were expected lile
# time_to_pressure, time_to_qb_hurry, coverage_assignment, and tackle_for_loss
print(df_player_play.isna().mean().sort_values(ascending=False).to_string())

# Checked the dataframe for any duplicated games, plays, or nfl players and found 0
df_player_play.duplicated(subset=['game_id', 'play_id', 'nfl_id']).sum()


game_id=df_player_play['game_id'].astype(str) #Converted game_id to a str
year=game_id.str[:4].astype(int) # Extracted the year from the game_id and converted to integer
month=game_id.str[4:6].astype(int) # Extracted the month from the game_id and converted to integer
df_player_play['season']=year.where(month > 2, year-1) #Using a where to define season, where if month > 2, we include the nfl game in th previous season


# For each player and season, created a new dataframe that calculates number of plays, the total tackles for loss and the average time to pressure as well
df_play_summary=df_player_play.groupby(['nfl_id','season']).agg(plays=('play_id', 'count',),
                                                                tackles_for_loss=('tackle_for_loss','sum'),
                                                                avg_time_to_pressure=('time_to_pressure','mean')).reset_index() 
df_play_summary.duplicated(subset=['nfl_id','season']).sum() # Found no duplicate values here at all after subsetting for nfl and season
df_play_summary.describe() # Generated summary stats and determined no weird outliers could skew the data
# Converted to CSV file as df_play_summary
df_play_summary.to_sql(name='df_play_summary', con=con, if_exists="replace", index=False) # Converted dataframe to SQL db


rush_yards                               0.999930
fumble                                   0.999768
fumble_lost                              0.999768
recovered_fumble                         0.992953
quick_pressure                           0.988023
time_to_pressure                         0.987893
time_to_qb_hurry                         0.987654
time_to_pressure_allowed                 0.984182
rec_yards                                0.980076
yards_after_catch                        0.980076
expected_yards_after_catch               0.971432
assist                                   0.962138
tackle                                   0.962138
caused_forced_fumble                     0.962138
tackle_for_loss                          0.962138
penalty_yards                            0.933959
player_get_off                           0.859954
separation_at_pass_forward               0.851249
route_ran                                0.830336
peak_pressure_probability_allowed        0.815313


922